# GPT OSS 120B — No-Tool Math SFT

Fine-tune `openai/gpt-oss-120b` via Tinker on DeepSeek-style high-reasoning math traces.

**Key Details:**
- Uses `GptOssRenderer` (Harmony format) — NOT manual `build_datum_manual`
- Converts `<think>...</think>` blocks → Harmony `ThinkingPart`/`TextPart`
- LoRA rank=32, cosine LR schedule, 1 epoch
- Dataset: Nemotron-Cascade no-tool math (competition-style `\boxed` problems)

In [ ]:
# ============================================================
# Cell 1: Configuration
# ============================================================

class Config:
    model_name = "openai/gpt-oss-120b"
    lora_rank = 32
    alpha = 64
    lr = 2e-4
    epoch = 1
    batch_size = 32          # 120B model — smaller batches than ref (64)
    max_length = 100_000     # Very long ctx for deep reasoning traces
    warmup_ratio = 0.05      # 5% linear warmup
    lr_schedule = 'cosine'   # Cosine decay after warmup
    eval_split = 5           # Hold out 5 examples for eval
    save_every = 50          # Checkpoint every 50 steps
    adam_beta1 = 0.9
    adam_beta2 = 0.95
    adam_eps = 1e-8

print("Config loaded:")
for k, v in vars(Config).items():
    if not k.startswith('_'):
        print(f"  {k}: {v}")

In [ ]:
# ============================================================
# Cell 2: Install Dependencies
# ============================================================

!pip install -q tinker tinker-cookbook transformers safetensors requests

In [ ]:
# ============================================================
# Cell 3: Imports & API Key
# ============================================================

import os, json, time, random, logging, re, ast, math
import pandas as pd
import tinker
from tinker_cookbook.renderers.gpt_oss import GptOssRenderer
from tinker_cookbook.supervised.common import datum_from_model_input_weights
from tinker_cookbook import tokenizer_utils

# ---- SET YOUR TINKER API KEY HERE ----
os.environ["TINKER_API_KEY"] = "YOUR_API_KEY_HERE"  # <-- REPLACE THIS!

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("gpt-oss-notool-sft")

print(f"Tinker SDK version: {tinker.__version__}")
print(f"API key set: {'TINKER_API_KEY' in os.environ and os.environ['TINKER_API_KEY'] != 'YOUR_API_KEY_HERE'}")

In [ ]:
# ============================================================
# Cell 4: Load Dataset
# ============================================================

import pandas as pd

# Load the JSONL dataset from Kaggle input
data = pd.read_json(
    "/kaggle/input/datasets/nahidhossainredom/nemotron-cascade-math-tool-no-tool-10k-rows/math_notool_10k.jsonl",
    lines=True
)

# Filter: keep only rows where user prompt contains \boxed (competition-style)
user_contents = data['messages'].apply(lambda x: x[1]['content'])
mask = user_contents.str.contains(r"\\boxed", case=False, regex=False)

# Separate the data
boxed_data = data[mask].copy()
non_boxed_data = data[~mask].copy()
training_data = boxed_data.copy()

print(f"Total rows: {len(data)}")
print(f"After \\boxed filter: {len(training_data)} (dropped {len(non_boxed_data)})")
print(f"Sample row keys: {list(training_data.iloc[0].keys())}")

In [ ]:
# ============================================================
# Cell 5: Inspect Raw Data
# ============================================================

sample = training_data.iloc[0]
msgs = sample['messages']

# Handle messages stored as string (from CSV/JSONL)
if isinstance(msgs, str):
    msgs = ast.literal_eval(msgs)

print(f"Number of messages: {len(msgs)}")
print(f"{'='*60}")
for i, msg in enumerate(msgs):
    content_preview = str(msg.get('content', ''))[:200]
    print(f"  [{i}] role={msg['role']}:")
    print(f"       {content_preview}...")
    print()

In [ ]:
# ============================================================
# Cell 6: DeepSeek → Harmony Formatter (No-Tool)
# ============================================================

def format_notool_to_harmony(messages: list) -> list:
    """
    Convert DeepSeek no-tool conversation to GPT OSS Harmony format.
    
    - Strips system messages (contain tool definitions we don't need)
    - Converts <think>...</think> blocks → {"type": "thinking", "thinking": "..."}
    - Text after </think> → {"type": "text", "text": "..."}
    
    The GptOssRenderer then maps:
      thinking → <|start|>assistant<|channel|>analysis<|message|>...<|end|>
      text     → <|start|>assistant<|channel|>final<|message|>...<|return|>
    """
    formatted = []
    for msg in messages:
        # Skip system messages (tool definitions not needed for no-tool)
        if msg["role"] == "system":
            continue
        
        if msg["role"] == "assistant":
            content = msg["content"]
            think_match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
            
            parts = []
            if think_match:
                thinking_text = think_match.group(1).strip()
                if thinking_text:
                    parts.append({"type": "thinking", "thinking": thinking_text})
                after_think = content[think_match.end():].strip()
                if after_think:
                    parts.append({"type": "text", "text": after_think})
            else:
                # No <think> block — treat entire content as final text
                parts.append({"type": "text", "text": content.strip()})
            
            formatted.append({"role": "assistant", "content": parts})
        else:
            # User messages pass through unchanged
            formatted.append(msg)
    
    return formatted


# Quick test
test_msgs = training_data.iloc[0]['messages']
if isinstance(test_msgs, str):
    test_msgs = ast.literal_eval(test_msgs)
test_harmony = format_notool_to_harmony(test_msgs)

print(f"Original messages: {len(test_msgs)} → Harmony messages: {len(test_harmony)}")
for i, msg in enumerate(test_harmony):
    if msg['role'] == 'assistant':
        print(f"  [{i}] assistant parts: {[p['type'] for p in msg['content']]}")
        for p in msg['content']:
            preview = p.get('thinking', p.get('text', ''))[:100]
            print(f"        {p['type']}: {preview}...")
    else:
        print(f"  [{i}] {msg['role']}: {str(msg['content'])[:100]}...")

In [ ]:
# ============================================================
# Cell 7: Initialize Tokenizer & Renderer
# ============================================================

tokenizer = tokenizer_utils.get_tokenizer(Config.model_name)
print(f"Tokenizer loaded! Vocab size: {tokenizer.vocab_size}")

renderer = GptOssRenderer(
    tokenizer=tokenizer,
    use_system_prompt=True,
    reasoning_effort="high",       # High reasoning for math
    current_date="2026-03-29",     # Fixed for reproducibility
)

# Quick test: render one sample
sample_msgs = training_data.iloc[0]['messages']
if isinstance(sample_msgs, str):
    sample_msgs = ast.literal_eval(sample_msgs)
sample_formatted = format_notool_to_harmony(sample_msgs)

model_input, weights = renderer.build_supervised_example(sample_formatted)
print(f"Token count: {model_input.length}")
print(f"Trainable tokens (weight>0): {int(weights.sum().item())}")
print(f"Context tokens (weight=0): {model_input.length - int(weights.sum().item())}")
print("\n✓ Renderer working!")

In [ ]:
# ============================================================
# Cell 8: Convert All Rows to Datums
# ============================================================

def row_to_datum(row):
    """Convert one dataset row into a Tinker Datum."""
    msgs = row['messages']
    if isinstance(msgs, str):
        msgs = ast.literal_eval(msgs)
    harmony_msgs = format_notool_to_harmony(msgs)
    model_input, weights = renderer.build_supervised_example(harmony_msgs)
    return datum_from_model_input_weights(model_input, weights, max_length=Config.max_length)


print(f"Converting {len(training_data)} rows to datums...")
all_datums = []
skipped = 0
t0 = time.time()

for i, (_, row) in enumerate(training_data.iterrows()):
    try:
        datum = row_to_datum(row)
        all_datums.append(datum)
    except Exception as e:
        skipped += 1
        if skipped <= 5:
            print(f"  ⚠ Skipped row {i}: {e}")
    if (i + 1) % 1000 == 0:
        elapsed = time.time() - t0
        print(f"  Processed {i+1}/{len(training_data)}... ({elapsed:.1f}s)")

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"Converted {len(all_datums)} datums ({skipped} skipped) in {elapsed:.1f}s")

# Token stats
token_counts = [d.model_input.length for d in all_datums]
print(f"Token stats: mean={sum(token_counts)/len(token_counts):.0f}, "
      f"max={max(token_counts)}, min={min(token_counts)}")

In [ ]:
# ============================================================
# Cell 9: Sanity Checks
# ============================================================

print("=== Sanity Check: Decoded tokens for 3 random samples ===\n")

random.seed(123)
for idx in random.sample(range(len(all_datums)), min(3, len(all_datums))):
    datum = all_datums[idx]
    tokens = []
    for chunk in datum.model_input.chunks:
        tokens.extend(chunk.tokens)
    decoded = tokenizer.decode(tokens)
    
    # Verify Harmony special tokens are present
    has_start = "<|start|>" in decoded
    has_channel = "<|channel|>" in decoded
    has_message = "<|message|>" in decoded
    has_end = "<|end|>" in decoded or "<|return|>" in decoded
    
    print(f"Sample {idx}: {len(tokens)} tokens")
    print(f"  Has <|start|>: {has_start}, <|channel|>: {has_channel}, "
          f"<|message|>: {has_message}, <|end|>/<|return|>: {has_end}")
    
    # Check training weight distribution
    w = datum.loss_fn_inputs["weights"].data
    n_train = sum(1 for x in w if x > 0)
    print(f"  Trainable tokens: {n_train}/{len(w)} ({100*n_train/max(len(w),1):.1f}%)")
    print(f"  First 300 chars: {decoded[:300]}...")
    print()

print("✓ Sanity checks complete!")

In [ ]:
# ============================================================
# Cell 10: Training Loop
# ============================================================

def compute_nll(fwd_bwd_result, batch):
    """Compute mean negative log-likelihood from forward-backward results."""
    total_nll = 0.0
    total_weight = 0.0
    for output, datum in zip(fwd_bwd_result.loss_fn_outputs, batch):
        logprobs = output["logprobs"].data
        w = datum.loss_fn_inputs["weights"].data
        for lp, wi in zip(logprobs, w):
            if wi > 0:
                total_nll -= lp
                total_weight += wi
    return total_nll / max(total_weight, 1.0)


def train_lora(all_datums):
    """Full LoRA SFT training loop on Tinker for GPT OSS 120B."""
    cfg = Config
    
    # ---- Train / Eval split ----
    random.seed(42)
    indices = list(range(len(all_datums)))
    random.shuffle(indices)
    eval_size = min(cfg.eval_split, len(all_datums) // 10)
    eval_datums = [all_datums[i] for i in indices[:eval_size]]
    train_datums = [all_datums[i] for i in indices[eval_size:]]
    
    n_batches_per_epoch = len(train_datums) // cfg.batch_size
    total_steps = n_batches_per_epoch * cfg.epoch
    warmup_steps = max(1, int(cfg.warmup_ratio * total_steps))
    
    print(f"╔{'═'*58}╗")
    print(f"║  GPT OSS 120B No-Tool SFT Training")
    print(f"╠{'═'*58}╣")
    print(f"║  Train samples: {len(train_datums):>8}  │  Eval samples: {len(eval_datums):>5}")
    print(f"║  Batch size:    {cfg.batch_size:>8}  │  Steps/epoch:  {n_batches_per_epoch:>5}")
    print(f"║  Total steps:   {total_steps:>8}  │  Warmup steps: {warmup_steps:>5}")
    print(f"║  LR: {cfg.lr:.1e}  │  Schedule: {cfg.lr_schedule}")
    print(f"║  LoRA rank: {cfg.lora_rank}  │  Save every: {cfg.save_every} steps")
    print(f"╚{'═'*58}╝")
    
    # ---- Connect to Tinker ----
    print("\n⏳ Connecting to Tinker API...")
    service_client = tinker.ServiceClient()
    training_client = service_client.create_lora_training_client(
        base_model=cfg.model_name,
        rank=cfg.lora_rank,
    )
    print("✓ Connected!\n")
    
    # ---- Training loop ----
    global_step = 0
    best_eval_nll = float("inf")
    best_checkpoint_path = None
    train_losses = []
    t_start = time.time()
    
    for epoch in range(cfg.epoch):
        epoch_indices = list(range(len(train_datums)))
        random.seed(epoch)
        random.shuffle(epoch_indices)
        
        for batch_idx in range(n_batches_per_epoch):
            step_start = time.time()
            
            # -- LR schedule: linear warmup → cosine decay --
            if global_step < warmup_steps:
                lr_mult = global_step / warmup_steps
            else:
                progress = (global_step - warmup_steps) / max(1, total_steps - warmup_steps)
                lr_mult = 0.5 * (1.0 + math.cos(math.pi * progress))
            current_lr = cfg.lr * lr_mult
            
            adam_params = tinker.AdamParams(
                learning_rate=current_lr,
                beta1=cfg.adam_beta1,
                beta2=cfg.adam_beta2,
                eps=cfg.adam_eps,
            )
            
            # -- Get batch --
            start_idx = batch_idx * cfg.batch_size
            batch = [train_datums[epoch_indices[i]]
                     for i in range(start_idx, start_idx + cfg.batch_size)]
            
            # -- Forward + Backward + Optimizer step --
            fwd_bwd_future = training_client.forward_backward(batch, loss_fn="cross_entropy")
            optim_future = training_client.optim_step(adam_params)
            fwd_bwd_result = fwd_bwd_future.result()
            optim_result = optim_future.result()
            
            train_nll = compute_nll(fwd_bwd_result, batch)
            train_losses.append(train_nll)
            step_time = time.time() - step_start
            
            # -- Log every 5 steps --
            if global_step % 5 == 0 or global_step == total_steps - 1:
                elapsed = time.time() - t_start
                avg_loss = sum(train_losses[-10:]) / len(train_losses[-10:])
                print(f"[Step {global_step:4d}/{total_steps}] "
                      f"epoch={epoch+1}/{cfg.epoch} lr={current_lr:.2e} "
                      f"train_nll={train_nll:.4f} avg_nll(10)={avg_loss:.4f} "
                      f"step={step_time:.1f}s elapsed={elapsed:.0f}s")
            
            # -- Evaluate periodically --
            if eval_datums and (global_step % cfg.save_every == 0 or global_step == total_steps - 1):
                eval_future = training_client.forward_backward(
                    eval_datums, loss_fn="cross_entropy")
                eval_result = eval_future.result()
                eval_nll = compute_nll(eval_result, eval_datums)
                is_best = eval_nll < best_eval_nll
                if is_best:
                    best_eval_nll = eval_nll
                print(f"  >>> EVAL nll={eval_nll:.4f} "
                      f"{'🏆 NEW BEST!' if is_best else ''} (best={best_eval_nll:.4f})")
                
                # Zero out gradient from eval forward-backward pass
                training_client.optim_step(
                    tinker.AdamParams(learning_rate=0.0, beta1=0.9, beta2=0.95, eps=1e-8)
                ).result()
            
            # -- Save checkpoint --
            if cfg.save_every > 0 and global_step % cfg.save_every == 0 and global_step > 0:
                name = f"step_{global_step:04d}"
                training_client.save_state(name=name).result()
                print(f"  💾 Saved checkpoint: {name}")
            
            global_step += 1
    
    # ---- Save final checkpoint ----
    print("\n⏳ Saving final checkpoint...")
    training_client.save_state(name="final").result()
    sampler_result = training_client.save_weights_for_sampler(
        name="gpt_oss_120b_notool_final").result()
    sampler_path = sampler_result.path
    
    total_time = time.time() - t_start
    print(f"\n{'='*60}")
    print(f"  ✅ TRAINING COMPLETE!")
    print(f"  Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
    print(f"  Final train NLL: {train_losses[-1]:.4f}")
    print(f"  Best eval NLL:   {best_eval_nll:.4f}")
    print(f"  Sampler path:    {sampler_path}")
    print(f"{'='*60}")
    
    return service_client, training_client, sampler_path, train_losses


# ============================================================
#  ▶ TO ACTUALLY TRAIN, UNCOMMENT THE LINE BELOW:
# ============================================================
# service_client, training_client, sampler_path, losses = train_lora(all_datums)
print("Training function defined! Uncomment the line above to run.")

In [ ]:
# ============================================================
# Cell 11: Evaluation (Sampling)
# ============================================================

def extract_boxed_balanced(text):
    """Extract content from the last \\boxed{...} in text, handling nested braces."""
    key = r"\boxed{"
    idx = text.rfind(key)
    if idx < 0:
        return None
    i = idx + len(key)
    depth = 1
    while i < len(text) and depth:
        if text[i] == "{": depth += 1
        elif text[i] == "}": depth -= 1
        i += 1
    return text[idx + len(key):i - 1].strip() if depth == 0 else None


def evaluate_model(service_client, sampler_path, test_problems, max_tokens=8192):
    """Run inference on test problems and check \\boxed accuracy."""
    from tinker.types import SamplingParams
    
    sampling_client = service_client.create_sampling_client(model_path=sampler_path)
    stop_sequences = renderer.get_stop_sequences()
    params = SamplingParams(max_tokens=max_tokens, temperature=0.6, stop=stop_sequences)
    
    correct = 0
    total = len(test_problems)
    
    for i, problem in enumerate(test_problems):
        messages = [{"role": "user", "content": problem["question"]}]
        prompt = renderer.build_generation_prompt(messages)
        output = sampling_client.sample(prompt, sampling_params=params, num_samples=1).result()
        
        response_msg, success = renderer.parse_response(output.sequences[0].tokens)
        
        # Extract text from structured content
        if isinstance(response_msg["content"], list):
            answer_text = "".join(
                p.get("text", "") for p in response_msg["content"] if p["type"] == "text"
            )
        else:
            answer_text = response_msg["content"]
        
        predicted = extract_boxed_balanced(answer_text)
        gold = str(problem["answer"]).strip()
        is_correct = predicted is not None and (
            predicted.strip() == gold or
            predicted.replace(" ", "") == gold.replace(" ", "")
        )
        if is_correct:
            correct += 1
        
        # Print first 5 results
        if i < 5:
            print(f"[{i+1}/{total}] Q: {problem['question'][:80]}...")
            print(f"  Predicted: {predicted}")
            print(f"  Gold:      {gold}")
            print(f"  Correct:   {'✓' if is_correct else '✗'}")
            print()
    
    acc = 100 * correct / total if total > 0 else 0
    print(f"\n{'='*40}")
    print(f"  Accuracy: {correct}/{total} = {acc:.1f}%")
    print(f"{'='*40}")
    return correct, total


# ============================================================
#  ▶ TO EVALUATE, UNCOMMENT:
# ============================================================
# evaluate_model(service_client, sampler_path, your_test_problems)
print("Evaluation function defined! Uncomment to run after training.")

In [ ]:
# ============================================================
# Cell 12: Download Weights
# ============================================================

def download_weights(service_client, sampler_path, output_dir="gpt_oss_120b_notool_weights"):
    """Download the trained LoRA weights from Tinker."""
    import requests
    
    os.makedirs(output_dir, exist_ok=True)
    rest_client = service_client.create_rest_client()
    url_resp = rest_client.get_checkpoint_archive_url_from_tinker_path(sampler_path).result()
    
    print(f"Downloading checkpoint from: {sampler_path}")
    r = requests.get(url_resp.url, stream=True)
    r.raise_for_status()
    total_bytes = int(r.headers.get('content-length', 0))
    print(f"  File size: {total_bytes / 1e9:.2f} GB")
    
    output_file = os.path.join(output_dir, "lora_checkpoint.tar")
    downloaded = 0
    with open(output_file, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total_bytes:
                pct = 100 * downloaded / total_bytes
                print(f"\r  Progress: {pct:.1f}% ({downloaded/1e9:.2f}/{total_bytes/1e9:.2f} GB)", end="")
    
    print(f"\n  ✓ Saved: {output_file} ({downloaded / 1e9:.2f} GB)")
    return output_file


# ============================================================
#  ▶ TO DOWNLOAD WEIGHTS, UNCOMMENT:
# ============================================================
# download_weights(service_client, sampler_path)
print("Download function defined! Uncomment to run after training.")